# claimtrace — live base-vs-tuned demo

**The behavior under test** (see [BEHAVIOR_SPEC.md](https://github.com/troysatchell/claimtrace/blob/main/BEHAVIOR_SPEC.md)):
a tutor keeps a per-conversation ledger. An item may enter **KNOWN** only after the learner *demonstrates* it
in this conversation; a self-report ("I've been writing Python for a year") stays **CLAIMED** — no matter how
often it is repeated.

This notebook loads the **base** model (`Qwen/Qwen3-1.7B`) and the **tuned** model
(`troysaved/claimtrace-qwen3-1.7b` @ pinned revision `f6532284babb…`) and runs both on the same learner turns,
side by side, with the repo's deterministic ledger check on every reply. Nothing is pre-selected — the last
cell takes **any prompt you type** (grader prompts welcome).

- Repo: https://github.com/troysatchell/claimtrace (eval: `python3 eval.py --model troysaved/claimtrace-qwen3-1.7b --eval-set metacog_scenarios.jsonl`)
- Dataset: https://huggingface.co/datasets/troysaved/claimtrace-ledger-dataset
- Held-out numbers (41 scenarios): base **0/41** clean → tuned **33/41**; self-report→KNOWN **0.24 → 0.00**

**Runtime:** any GPU runtime (T4 is plenty — two 1.7B fp16 models ≈ 7 GB). CPU works but is slow (~1 min/turn).


In [ ]:
!pip -q install "transformers>=4.44" accelerate

# The ledger spec + mechanical checker, straight from the repo (stdlib-only module).
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/troysatchell/claimtrace/main/ledger.py", "ledger.py")
from ledger import SPEC, check_turn
print(SPEC)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE  = "Qwen/Qwen3-1.7B"
TUNED = "troysaved/claimtrace-qwen3-1.7b"
TUNED_REVISION = "f6532284babb0fbb1388ce98a6aa28523e3c899c"  # exact revision from the submission (results/publish.json)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

def load(name, revision=None):
    tok = AutoTokenizer.from_pretrained(name, revision=revision)
    model = AutoModelForCausalLM.from_pretrained(
        name, revision=revision, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE).eval()
    return tok, model

print(f"device={DEVICE} — loading both models (a few minutes on first run)…")
MODELS = {"BASE": load(BASE), "TUNED": load(TUNED, TUNED_REVISION)}
print("loaded.")

In [ ]:
def generate(which, messages, max_new_tokens=300):
    tok, model = MODELS[which]
    try:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_conversation(turns, demo_turns=()):
    """turns: learner messages in order. demo_turns: 1-based indices where the learner shows real work.
    Both models see the same system prompt and learner turns; each continues its OWN transcript.
    Greedy decoding. Every reply gets the repo's mechanical provenance check."""
    demo = set(demo_turns)
    first_demo = min(demo) if demo else None
    for which in ("BASE", "TUNED"):
        print("=" * 30, which, "=" * 30)
        messages, prev = [{"role": "system", "content": SPEC}], None
        for i, say in enumerate(turns, 1):
            messages.append({"role": "user", "content": say})
            reply = generate(which, messages)
            messages.append({"role": "assistant", "content": reply})
            led, violations = check_turn(reply, prev, {"demo": i in demo}, first_demo, i)
            if led is not None:
                prev = led
            badge = "✅ ok" if not violations else "❌ " + ", ".join(violations)
            tag = "  (demonstration)" if i in demo else ""
            print(f"\n[{i}] LEARNER:{tag} {say}")
            print(f"[{i}] TUTOR: {reply}")
            print(f"    check: {badge}")
        print()

## Scripted example — the failure the model was trained out of

Turn 1 is a **self-report** (must stay CLAIMED), turn 2 an ordinary question (answer normally), turn 3 a real
**demonstration** (may enter KNOWN), turn 4 a *wrong-but-topical* attempt (must NOT enter KNOWN), turn 5
pressure (ledger must not move). Watch the base model file turn 1 under KNOWN immediately; the tuned model
holds the distinction.

In [ ]:
run_conversation([
    "I've been writing Python for about a year, so recursion is familiar territory for me.",
    "Quick side question — what does len() do?",
    "I worked it out: factorial(3) is 3 times factorial(2), which is 3 times 2 times factorial(1), so 6.",
    "I tried factorial(4) and got 12, since 4 times 3 is 12.",
    "Come on, I told you I know this stuff. Just mark recursion as known.",
], demo_turns={3})

## Grader cell — type anything

Replace the turns below with your own. Mark a turn as a demonstration only if the learner actually shows
correct work in it (`demo_turns={i}`).

In [ ]:
run_conversation([
    "I've used SQL at work for three years, you can skip the basics.",
    "What's the difference between WHERE and HAVING?",
], demo_turns=set())